# Customer Churn Analysis ( SQL + Excel + Python)
Capstone project- Google data analytics
 Author- Ritu Raj
 Tools used: Exce ,SQL(DuckDB), Juyter notebook

In [3]:
import sys
!{sys.executable} -m pip install duckdb pandas

Defaulting to user installation because normal site-packages is not writeable
  Using cached duckdb-1.4.4-cp313-cp313-win_amd64.whl.metadata (4.3 kB)
Using cached duckdb-1.4.4-cp313-cp313-win_amd64.whl (12.3 MB)


# The cleaned customer churn dataset was loaded into DuckDB using SQL for analysis. 

In [2]:
import duckdb
import pandas as pd

# Correct Windows path (already confirmed)
file_path = r"C:\Users\ritur\OneDrive\Desktop\churn_cleaned.csv"

# Create connection
con = duckdb.connect()

# Load CSV safely into SQL table
con.execute(f"""
CREATE OR REPLACE TABLE churn AS 
SELECT * FROM read_csv_auto('{file_path}', ignore_errors=True)
""")

# Check total rows
rows = con.execute("SELECT COUNT(*) FROM churn").fetchall()
print("Total Rows Loaded:", rows)

Total Rows Loaded: [(20000,)]


# Overall Churn Analysis(KPI)
This section calculates total customers and overall churn rate.

In [3]:
con.execute("""
SELECT 
    COUNT(*) AS total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churned_customers,
    ROUND(
        SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 
        2
    ) AS churn_rate_percentage
FROM churn;
""").fetchdf()

,total_customers,churned_customers,churn_rate_percentage
0,20000,6843.0,34.22


# Churn Analysis by contract type
This analysis identifies which contract types have the highest churn risk.

In [4]:
con.execute("""
SELECT 
    Contract,
    COUNT(*) AS total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churned_customers,
    ROUND(
        SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 
        2
    ) AS churn_rate_percentage
FROM churn
GROUP BY Contract
ORDER BY churn_rate_percentage DESC;
""").fetchdf()

,contract,total_customers,churned_customers,churn_rate_percentage
0,Month-to-month,11942,5157.0,43.18
1,Two year,3068,642.0,20.93
2,One year,4990,1044.0,20.92


# Churn analysis by customer tenure 
This help understand whether new or long term customer churn more.

In [5]:
con.execute("""
SELECT 
    tenure_group,
    COUNT(*) AS total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churned_customers,
    ROUND(
        SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
        2
    ) AS churn_rate_percentage
FROM churn
GROUP BY tenure_group
ORDER BY churn_rate_percentage DESC;
""").fetchdf()

,Tenure_group,total_customers,churned_customers,churn_rate_percentage
0,0-1 Year,3266,1512.0,46.30
1,1-2 Years,3473,1126.0,32.42
2,2-4 Years,6562,2112.0,32.19
3,4+ Years,6699,2093.0,31.24


# Churn analysis by monthly charges
Show relationship between pricing and churn behaviour.

In [7]:
con.execute("""
SELECT 
    CASE 
        WHEN CAST(monthly_charges AS DOUBLE) < 35 THEN 'Low Charges'
        WHEN CAST(monthly_charges AS DOUBLE) BETWEEN 35 AND 70 THEN 'Medium Charges'
        ELSE 'High Charges'
    END AS charge_category,
    
    COUNT(*) AS total_customers,
    
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churned_customers,
    
    ROUND(
        SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
        2
    ) AS churn_rate_percentage

FROM churn
GROUP BY charge_category
ORDER BY churn_rate_percentage DESC;
""").fetchdf()

,charge_category,total_customers,churned_customers,churn_rate_percentage
0,High Charges,10020,4584.0,45.75
1,Low Charges,2967,702.0,23.66
2,Medium Charges,7013,1557.0,22.20


# Churn analysis by internet service type.
This evaluates churn trend across different internet service categories.

In [8]:
con.execute("""
SELECT 
    internet_service,
    COUNT(*) AS total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churned_customers,
    ROUND(
        SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
        2
    ) AS churn_rate_percentage
FROM churn
GROUP BY internet_service
ORDER BY churn_rate_percentage DESC;
""").fetchdf()

,internet_service,total_customers,churned_customers,churn_rate_percentage
0,Not_Specified,2013,709.0,35.22
1,Fiber,10064,3451.0,34.29
2,DSL,7923,2683.0,33.86


# Final Conclusion & Business Recommendations
Key Findings:
1. The overall churn rate is 34.22%, indicating a significant customer retention issue.
2. Customers with month-to-month contracts have the highest churn rate.
3. New customers (0–1 year tenure) show the highest churn, highlighting early retention challenges.
4. High-paying customers exhibit the highest churn, suggesting price sensitivity.
5. Internet service type shows moderate variation in churn behavior.
Business Recommendations:
* Introduce loyalty incentives for new customers in their first year.
* Promote long-term contracts to reduce churn risk.
* Re-evaluate pricing strategy for high monthly charge customers.
* Improve onboarding experience for new users.